# Borough In or Out: Testing Geographic Bias vs. Filter-Only Retrieval

**Experiment Objective:**
Compare two embedding designs for NYC 311 heat/hot-water case retrieval:
- **Condition A (Current `resolveiq-cases-collection-00`):** `borough` is concatenated directly into the embedded text.
- **Condition B (Proposed `resolveiq-cases-collection-01`):** `borough` is excluded from the embedded text and used strictly as an exact Qdrant payload filter.

**Ground Rules:**
- Offline data loading from the existing `heat_2024_01_27_day.json` extract (798 records).
- No new external data downloads.
- No street addresses, latitude/longitude, or `incident_zip` exposed.
- Calls to Gemini embedding APIs or Qdrant collection creation occur only in designated sections with explicit confirmation.

In [1]:
import json
import hashlib
from pathlib import Path
import pandas as pd
from resolveiq.config import REPO_ROOT, get_settings

settings = get_settings()

raw_file = REPO_ROOT / "data" / "raw" / "heat_2024_01_27_day.json"
index_meta_file = REPO_ROOT / "data" / "indexes" / "resolveiq-cases-collection-00.meta.json"

print(f"Repo root:       {REPO_ROOT}")
print(f"Raw data file:   {raw_file.exists()} -> {raw_file.name} ({raw_file.stat().st_size:,} bytes)")
print(f"Collection meta: {index_meta_file.exists()} -> {index_meta_file.name}")
print(f"Embedding model: {settings.gemini_embed_model} ({settings.embedding_dimensions} dims)")
print(f"Qdrant target:   {settings.qdrant_url} (default collection: {settings.qdrant_collection})")

Repo root:       C:\Users\mejdi\Documents\resolveiq
Raw data file:   True -> heat_2024_01_27_day.json (662,653 bytes)
Collection meta: True -> resolveiq-cases-collection-00.meta.json
Embedding model: gemini-embedding-001 (3072 dims)
Qdrant target:   http://localhost:6333 (default collection: resolveiq-cases-collection-00)


In [2]:
with open(raw_file, "r", encoding="utf-8") as f:
    raw_records = json.load(f)

df = pd.DataFrame(raw_records)
print(f"Loaded {len(df)} records from {raw_file.name}\n")

# Condition A (5 fields: includes borough)
fields_a = ["complaint_type", "location_type", "descriptor", "descriptor_2", "borough"]
def build_text_a(row):
    return ", ".join([str(row[f]).strip() for f in fields_a if pd.notna(row[f]) and str(row[f]).strip()])

df["text_a"] = df.apply(build_text_a, axis=1)
shapes_a = sorted(df["text_a"].unique())

# Condition B (4 fields: borough excluded)
fields_b = ["complaint_type", "location_type", "descriptor", "descriptor_2"]
def build_text_b(row):
    return ", ".join([str(row[f]).strip() for f in fields_b if pd.notna(row[f]) and str(row[f]).strip()])

df["text_b"] = df.apply(build_text_b, axis=1)
shapes_b = sorted(df["text_b"].unique())

print(f"Condition A (with borough):    {len(shapes_a)} distinct text shapes across {len(df)} tickets")
print(f"Condition B (without borough): {len(shapes_b)} distinct complaint shapes across {len(df)} tickets\n")

print("Distinct Condition B complaint shapes:")
for i, s in enumerate(shapes_b, 1):
    count = (df["text_b"] == s).sum()
    print(f"  {i:2d}. [{count:3d} tickets] {s}")

Loaded 798 records from heat_2024_01_27_day.json

Condition A (with borough):    30 distinct text shapes across 798 tickets
Condition B (without borough): 6 distinct complaint shapes across 798 tickets

Distinct Condition B complaint shapes:
   1. [138 tickets] HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT
   2. [ 66 tickets] HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT AND NO HOT WATER
   3. [ 43 tickets] HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HOT WATER
   4. [280 tickets] HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT
   5. [220 tickets] HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT AND NO HOT WATER
   6. [ 51 tickets] HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HOT WATER


## Section 2: Exploratory Pilot Queries & Target Complaint Shapes

This exploratory pilot observes how natural language complaint prose behaves when mapped against the 6 canonical intake shapes across eight specific test queries.

**Evaluation Metric:**
- **Shape Hit@1:** Top-1 retrieved shape matches the manually authored target shape.
- **Shape Hit@3:** Target shape is in the top-3 retrieved shapes.

**Query Design:**
- Queries 1–4 are **borough-neutral** (no borough mentioned).
- Queries 5–8 are **borough-explicit** (specific borough mentioned).

In [3]:
# 1. Canonical 6 complaint shapes with stable IDs
SHAPE_CATALOG = {
    1: {"slug": "APT_NO_HEAT", "shape": "HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT"},
    2: {"slug": "APT_BOTH",    "shape": "HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT AND NO HOT WATER"},
    3: {"slug": "APT_NO_HOT",  "shape": "HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HOT WATER"},
    4: {"slug": "BLDG_NO_HEAT", "shape": "HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT"},
    5: {"slug": "BLDG_BOTH",    "shape": "HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT AND NO HOT WATER"},
    6: {"slug": "BLDG_NO_HOT",  "shape": "HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HOT WATER"},
}

catalog_df = pd.DataFrame(SHAPE_CATALOG).T
catalog_df.index.name = "shape_id"

# 2. Eight exploratory triage pilot queries
PILOT_QUERIES = [
    # Borough-neutral queries (testing pure complaint shape mapping and borough ranking bias)
    {
        "qid": "Q1",
        "text": "the radiators stayed cold overnight in my apartment",
        "explicit_borough": None,
        "target_shape_id": 1,
        "target_slug": "APT_NO_HEAT"
    },
    {
        "qid": "Q2",
        "text": "my shower has zero hot water this morning but the heat is on fine",
        "explicit_borough": None,
        "target_shape_id": 3,
        "target_slug": "APT_NO_HOT"
    },
    {
        "qid": "Q3",
        "text": "the whole building has no heat or hot water and the super is not responding",
        "explicit_borough": None,
        "target_shape_id": 5,
        "target_slug": "BLDG_BOTH"
    },
    {
        "qid": "Q4",
        "text": "all tenants in the complex have cold radiators since yesterday",
        "explicit_borough": None,
        "target_shape_id": 4,
        "target_slug": "BLDG_NO_HEAT"
    },
    # Borough-explicit queries (testing filter separation vs text embedding)
    {
        "qid": "Q5",
        "text": "freezing cold unit in Brooklyn with no heat at all",
        "clean_complaint_text": "freezing cold unit with no heat at all",
        "explicit_borough": "BROOKLYN",
        "target_shape_id": 1,
        "target_slug": "APT_NO_HEAT"
    },
    {
        "qid": "Q6",
        "text": "boiler shut down in Bronx apartment building no heat for any floor",
        "clean_complaint_text": "boiler shut down in apartment building no heat for any floor",
        "explicit_borough": "BRONX",
        "target_shape_id": 4,
        "target_slug": "BLDG_NO_HEAT"
    },
    {
        "qid": "Q7",
        "text": "no hot water anywhere in my Queens apartment since Friday",
        "clean_complaint_text": "no hot water anywhere in my apartment since Friday",
        "explicit_borough": "QUEENS",
        "target_shape_id": 3,
        "target_slug": "APT_NO_HOT"
    },
    {
        "qid": "Q8",
        "text": "Manhattan high rise completely without heat and hot water",
        "clean_complaint_text": "high rise completely without heat and hot water",
        "explicit_borough": "MANHATTAN",
        "target_shape_id": 5,
        "target_slug": "BLDG_BOTH"
    }
]

pilot_df = pd.DataFrame(PILOT_QUERIES)
print(f"Authored {len(pilot_df)} exploratory pilot queries across {len(SHAPE_CATALOG)} canonical complaint shapes:\n")
print(pilot_df[["qid", "text", "explicit_borough", "target_slug"]].to_string(index=False))

Authored 8 exploratory pilot queries across 6 canonical complaint shapes:

qid                                                                        text explicit_borough  target_slug
 Q1                         the radiators stayed cold overnight in my apartment              NaN  APT_NO_HEAT
 Q2           my shower has zero hot water this morning but the heat is on fine              NaN   APT_NO_HOT
 Q3 the whole building has no heat or hot water and the super is not responding              NaN    BLDG_BOTH
 Q4              all tenants in the complex have cold radiators since yesterday              NaN BLDG_NO_HEAT
 Q5                          freezing cold unit in Brooklyn with no heat at all         BROOKLYN  APT_NO_HEAT
 Q6          boiler shut down in Bronx apartment building no heat for any floor            BRONX BLDG_NO_HEAT
 Q7                   no hot water anywhere in my Queens apartment since Friday           QUEENS   APT_NO_HOT
 Q8                   Manhattan high rise com

In [ ]:
import httpx
from qdrant_client import QdrantClient

# Set to True ONLY if you deliberately want to wipe and rebuild collection-01
FORCE_REBUILD = False

COLLECTION_00 = settings.qdrant_collection  # "resolveiq-cases-collection-00"
COLLECTION_01 = "resolveiq-cases-collection-01"

qdrant_live = False
c00_exists = False
c01_exists = False

try:
    resp = httpx.get(f"{settings.qdrant_url}/collections", timeout=2.0)
    if resp.status_code == 200:
        qdrant_live = True
        client_qdrant = QdrantClient(url=settings.qdrant_url)
        all_collections = [c.name for c in client_qdrant.get_collections().collections]
        
        print(f"Qdrant service: ONLINE at {settings.qdrant_url}")
        print(f"Collections:    {all_collections}")
        
        # Check Collection-00
        if COLLECTION_00 in all_collections:
            info_00 = client_qdrant.get_collection(COLLECTION_00)
            print(f"  * {COLLECTION_00}: {info_00.points_count} points")
            c00_exists = True
        else:
            print(f"  * {COLLECTION_00}: NOT FOUND")
            
        # Check Collection-01 and enforce fail-safe pre-API guard
        if COLLECTION_01 in all_collections:
            info_01 = client_qdrant.get_collection(COLLECTION_01)
            c01_count = info_01.points_count
            c01_dims = info_01.config.params.vectors.size
            print(f"  * {COLLECTION_01}: {c01_count} points (dims: {c01_dims}, expected: {len(df)} points @ {settings.embedding_dimensions} dims)")
            
            is_valid = (c01_count == len(df)) and (c01_dims == settings.embedding_dimensions)
            
            if is_valid and not FORCE_REBUILD:
                c01_exists = True
                COLLECTION_01_READY = True
                print(f"\n[RERUN MODE ACTIVE] '{COLLECTION_01}' exists and matches all expectations ({c01_count} points).")
                print("Downstream cells will use cached vectors and skip re-indexing.")
            elif not is_valid and not FORCE_REBUILD:
                raise RuntimeError(
                    f"SAFETY HALT: '{COLLECTION_01}' exists but does not match expectations "
                    f"({c01_count} points, dims: {c01_dims}; expected {len(df)} points @ {settings.embedding_dimensions} dims).\n"
                    "To prevent accidental data loss or unwanted API calls, manual review is required.\n"
                    "Set FORCE_REBUILD = True at the top of this cell only if you explicitly intend to delete and rebuild it."
                )
            else:
                c01_exists = False
                COLLECTION_01_READY = False
                print(f"\n[REBUILD MODE] FORCE_REBUILD=True: '{COLLECTION_01}' will be recreated in Section 5.")
        else:
            print(f"  * {COLLECTION_01}: NOT FOUND (ready for first build)")
            c01_exists = False
            COLLECTION_01_READY = False

    else:
        print(f"Qdrant returned HTTP {resp.status_code}: {resp.text}")
except RuntimeError:
    raise
except Exception as exc:
    print(f"Qdrant service: OFFLINE or UNREACHABLE at {settings.qdrant_url}")
    print(f"Reason:         {type(exc).__name__}: {exc}")
    print("\nTo start Qdrant via Docker Compose, run in your terminal:")
    print("  docker compose up -d qdrant")

In [5]:
import json
import time
from pathlib import Path
from google import genai

cache_path = REPO_ROOT / "data" / "indexes" / "pilot_vectors_cache.json"
cached_vectors = {}
if cache_path.exists():
    try:
        with open(cache_path, "r", encoding="utf-8") as f:
            cached_vectors = json.load(f)
    except Exception:
        cached_vectors = {}

pilot_embeddings = cached_vectors.get("pilot_embeddings", {})

if len(pilot_embeddings) == len(PILOT_QUERIES):
    print(f"Loaded {len(pilot_embeddings)} cached pilot query vectors from {cache_path.name} (0 Gemini API calls).")
else:
    ai_client = genai.Client(api_key=settings.gemini_api_key.get_secret_value())
    print(f"Embedding {len(PILOT_QUERIES)} pilot queries with {settings.gemini_embed_model}...")
    t0 = time.perf_counter()
    for q in PILOT_QUERIES:
        qid = q["qid"]
        response = ai_client.models.embed_content(
            model=settings.gemini_embed_model,
            contents=q["text"]
        )
        pilot_embeddings[qid] = response.embeddings[0].values
    print(f"Done in {time.perf_counter() - t0:.2f}s.")
    cached_vectors["pilot_embeddings"] = pilot_embeddings
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(cached_vectors, f)
    print(f"Cached {len(pilot_embeddings)} vectors to {cache_path.name}.")

dim_check = {len(v) for v in pilot_embeddings.values()}
print(f"Vectors ready:     {len(pilot_embeddings)}")
print(f"Dimensions:        {dim_check} (expected {settings.embedding_dimensions})")

Embedding 8 pilot queries with gemini-embedding-001...
Done in 66.87s.
Cached 8 vectors to pilot_vectors_cache.json.
Vectors ready:     8
Dimensions:        {3072} (expected 3072)


In [6]:
results_a = []

for q in PILOT_QUERIES:
    qid = q["qid"]
    target_shape = SHAPE_CATALOG[q["target_shape_id"]]["shape"]
    target_slug = q["target_slug"]
    qv = pilot_embeddings[qid]
    
    # Query Qdrant grouped by text_to_embed to avoid redundant duplicates of the same shape
    res = client_qdrant.query_points_groups(
        collection_name=settings.qdrant_collection,
        query=qv,
        group_by="text_to_embed",
        limit=5,
        group_size=1,
        with_payload=True
    )
    
    # Extract retrieved shapes and their top hits
    retrieved_groups = []
    for g in res.groups:
        top_hit = g.hits[0]
        full_text = g.id  # e.g. "HEAT/HOT WATER, ..., NO HEAT, BROOKLYN"
        
        # In Condition A, borough is the 5th comma-separated token
        parts = [p.strip() for p in full_text.split(",")]
        borough_token = parts[-1] if len(parts) >= 5 else ""
        complaint_only = ", ".join(parts[:-1]) if len(parts) >= 5 else full_text
        
        retrieved_groups.append({
            "complaint_shape": complaint_only,
            "borough": borough_token,
            "score": top_hit.score,
            "ticket": top_hit.id
        })
    
    # Evaluate metrics against the 6 core complaint shapes
    top1 = retrieved_groups[0] if retrieved_groups else None
    top3_shapes = [g["complaint_shape"] for g in retrieved_groups[:3]]
    
    hit_1 = (top1["complaint_shape"] == target_shape) if top1 else False
    hit_3 = (target_shape in top3_shapes)
    
    # Identify which slug was retrieved
    retrieved_slug = "UNKNOWN"
    for sid, info in SHAPE_CATALOG.items():
        if info["shape"] == (top1["complaint_shape"] if top1 else ""):
            retrieved_slug = info["slug"]
            break
            
    results_a.append({
        "qid": qid,
        "explicit_borough": q["explicit_borough"] or "—",
        "target": target_slug,
        "retrieved_top1": retrieved_slug,
        "top1_borough": top1["borough"] if top1 else "—",
        "score": f"{top1['score']:.4f}" if top1 else "—",
        "hit@1": "PASS" if hit_1 else "FAIL",
        "hit@3": "PASS" if hit_3 else "FAIL"
    })

eval_a_df = pd.DataFrame(results_a)
print("=== Condition A (Borough Embedded in Vector) Evaluation Results ===\n")
print(eval_a_df.to_string(index=False))

hit1_rate = (eval_a_df["hit@1"] == "PASS").mean()
hit3_rate = (eval_a_df["hit@3"] == "PASS").mean()
print(f"\nCondition A Summary: Shape Hit@1 = {hit1_rate:.1%} ({hit1_rate*len(eval_a_df):.0f}/{len(eval_a_df)}) | Shape Hit@3 = {hit3_rate:.1%} ({hit3_rate*len(eval_a_df):.0f}/{len(eval_a_df)})")

=== Condition A (Borough Embedded in Vector) Evaluation Results ===

qid explicit_borough       target retrieved_top1 top1_borough  score hit@1 hit@3
 Q1                —  APT_NO_HEAT    APT_NO_HEAT    MANHATTAN 0.7294  PASS  PASS
 Q2                —   APT_NO_HOT     APT_NO_HOT    MANHATTAN 0.7006  PASS  PASS
 Q3                —    BLDG_BOTH   BLDG_NO_HEAT    MANHATTAN 0.8509  FAIL  PASS
 Q4                — BLDG_NO_HEAT   BLDG_NO_HEAT       QUEENS 0.7653  PASS  PASS
 Q5         BROOKLYN  APT_NO_HEAT   BLDG_NO_HEAT     BROOKLYN 0.8422  FAIL  PASS
 Q6            BRONX BLDG_NO_HEAT   BLDG_NO_HEAT        BRONX 0.9043  PASS  PASS
 Q7           QUEENS   APT_NO_HOT    BLDG_NO_HOT       QUEENS 0.8747  FAIL  PASS
 Q8        MANHATTAN    BLDG_BOTH   BLDG_NO_HEAT    MANHATTAN 0.8944  FAIL  PASS

Condition A Summary: Shape Hit@1 = 50.0% (4/8) | Shape Hit@3 = 100.0% (8/8)


## Section 5: Condition B (`resolveiq-cases-collection-01`) — Decoupled Borough & Payload Filter

In Condition B:
1. **Embedding Text:** Formed strictly from 4 intake fields (`complaint_type`, `location_type`, `descriptor`, `descriptor_2`). Borough is entirely excluded from vector generation.
2. **Deterministic Grouping:** Each point is assigned a canonical `shape_id` (1–6).
3. **Structured Payload:** Individual complaint fields are stored cleanly in the payload. `incident_zip` is permanently omitted.
4. **Borough Filtering:** When an explicit borough is provided, it is applied solely as an exact Qdrant payload filter; the embedding text receives only the clean complaint description.

In [7]:
shape_vectors_raw = cached_vectors.get("shape_vectors", {})
clean_query_vectors = cached_vectors.get("clean_query_vectors", {})

if len(shape_vectors_raw) == len(SHAPE_CATALOG) and len(clean_query_vectors) == len(PILOT_QUERIES):
    shape_vectors = {int(k): v for k, v in shape_vectors_raw.items()}
    print(f"Loaded {len(shape_vectors)} shape vectors and {len(clean_query_vectors)} clean query vectors from {cache_path.name} (0 Gemini API calls).")
else:
    ai_client = genai.Client(api_key=settings.gemini_api_key.get_secret_value())
    print("Embedding 6 canonical complaint shapes for Condition B index...")
    t0 = time.perf_counter()
    shape_vectors = {}
    for sid, item in SHAPE_CATALOG.items():
        resp = ai_client.models.embed_content(
            model=settings.gemini_embed_model,
            contents=item["shape"]
        )
        shape_vectors[sid] = resp.embeddings[0].values
    print(f"6 shapes embedded in {time.perf_counter() - t0:.2f}s.")

    print("Embedding 4 clean complaint query texts for Q5-Q8...")
    t1 = time.perf_counter()
    clean_query_vectors = {}
    for q in PILOT_QUERIES:
        qid = q["qid"]
        if q["explicit_borough"] is not None:
            clean_text = q["clean_complaint_text"]
            resp = ai_client.models.embed_content(
                model=settings.gemini_embed_model,
                contents=clean_text
            )
            clean_query_vectors[qid] = resp.embeddings[0].values
        else:
            clean_query_vectors[qid] = pilot_embeddings[qid]
    print(f"Clean query vectors ready in {time.perf_counter() - t1:.2f}s.")

    cached_vectors["shape_vectors"] = {str(k): v for k, v in shape_vectors.items()}
    cached_vectors["clean_query_vectors"] = clean_query_vectors
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(cached_vectors, f)
    print(f"Cached all Condition B vectors to {cache_path.name}.")

Embedding 6 canonical complaint shapes for Condition B index...
6 shapes embedded in 2.64s.
Embedding 4 clean complaint query texts for Q5-Q8...
Clean query vectors ready in 1.64s.
Cached all Condition B vectors to pilot_vectors_cache.json.


In [ ]:
from qdrant_client.models import VectorParams, Distance, PointStruct

COLLECTION_01 = "resolveiq-cases-collection-01"

shape_text_to_id = {item["shape"]: sid for sid, item in SHAPE_CATALOG.items()}
df["shape_id"] = df["text_b"].map(shape_text_to_id)

collection_exists = client_qdrant.collection_exists(COLLECTION_01)
current_count = client_qdrant.count(COLLECTION_01, exact=True).count if collection_exists else 0

if collection_exists and not FORCE_REBUILD:
    if current_count == len(df):
        server_count = current_count
        print(f"Collection '{COLLECTION_01}' already exists and matches expected count ({server_count} points).")
        print("Skipping deletion and re-indexing (0 Qdrant writes). Set FORCE_REBUILD = True in Section 3 to recreate.")
    else:
        raise RuntimeError(
            f"SAFETY HALT: Collection '{COLLECTION_01}' exists with {current_count} points (expected {len(df)}), "
            f"but FORCE_REBUILD is False.\nAborting to prevent silent collection deletion."
        )
else:
    if collection_exists:
        print(f"FORCE_REBUILD is True: Deleting and recreating '{COLLECTION_01}'...")
        client_qdrant.delete_collection(COLLECTION_01)

    client_qdrant.create_collection(
        collection_name=COLLECTION_01,
        vectors_config=VectorParams(size=settings.embedding_dimensions, distance=Distance.COSINE)
    )
    print(f"Created collection '{COLLECTION_01}' in Qdrant.")

    points = []
    for _, row in df.iterrows():
        sid = int(row["shape_id"])
        vec = shape_vectors[sid]
        
        payload = {
            "unique_key": str(row.get("unique_key", "")),
            "shape_id": sid,
            "shape_slug": SHAPE_CATALOG[sid]["slug"],
            "complaint_type": str(row.get("complaint_type", "")),
            "location_type": str(row.get("location_type", "")),
            "descriptor": str(row.get("descriptor", "")),
            "descriptor_2": str(row.get("descriptor_2", "")),
            "borough": str(row.get("borough", "")).upper(),
            "created_date": str(row.get("created_date", "")),
            "status": str(row.get("status", "")),
            "closed_date": str(row.get("closed_date", "")) if pd.notna(row.get("closed_date")) else None,
            "resolution_description": str(row.get("resolution_description", "")) if pd.notna(row.get("resolution_description")) else None
        }
        
        points.append(PointStruct(
            id=int(row["unique_key"]),
            vector=vec,
            payload=payload
        ))

    BATCH_SIZE = 100
    t_up = time.perf_counter()
    for i in range(0, len(points), BATCH_SIZE):
        batch = points[i:i + BATCH_SIZE]
        client_qdrant.upsert(collection_name=COLLECTION_01, points=batch, wait=True)

    elapsed_up = time.perf_counter() - t_up
    server_count = client_qdrant.count(collection_name=COLLECTION_01, exact=True).count
    print(f"Successfully upserted {server_count} points into '{COLLECTION_01}' in {elapsed_up:.2f}s.")

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

# Strict serialization gateway: outcome fields are populated ONLY when closed_date < intake_time
def format_retrieved_case(payload: dict, intake_time: str) -> dict:
    c_date = payload.get("closed_date")
    is_historical = (c_date is not None) and (c_date < intake_time)
    
    return {
        "ticket": payload["unique_key"],
        "shape_id": payload["shape_id"],
        "shape_slug": payload["shape_slug"],
        "complaint_type": payload["complaint_type"],
        "location_type": payload["location_type"],
        "descriptor": payload["descriptor"],
        "descriptor_2": payload["descriptor_2"],
        "borough": payload["borough"],
        "created_date": payload["created_date"],
        "is_historical_evidence": is_historical,
        "status": payload["status"] if is_historical else None,
        "closed_date": payload["closed_date"] if is_historical else None,
        "resolution_description": payload["resolution_description"] if is_historical else None
    }

results_b = []

for q in PILOT_QUERIES:
    qid = q["qid"]
    target_sid = q["target_shape_id"]
    target_slug = q["target_slug"]
    qv = clean_query_vectors[qid]
    
    qfilter = None
    if q["explicit_borough"]:
        qfilter = Filter(must=[FieldCondition(key="borough", match=MatchValue(value=q["explicit_borough"].upper()))])
        
    res = client_qdrant.query_points_groups(
        collection_name=COLLECTION_01,
        query=qv,
        query_filter=qfilter,
        group_by="shape_id",
        limit=5,
        group_size=1,
        with_payload=True
    )
    
    retrieved_groups = []
    for g in res.groups:
        top_hit = g.hits[0]
        # format_retrieved_case is the SOLE gateway for returning point payloads
        formatted = format_retrieved_case(top_hit.payload, intake_time=top_hit.payload["created_date"])
        formatted["score"] = top_hit.score
        retrieved_groups.append(formatted)
        
    top1 = retrieved_groups[0] if retrieved_groups else None
    top3_sids = [g["shape_id"] for g in retrieved_groups[:3]]
    
    hit_1 = (top1["shape_id"] == target_sid) if top1 else False
    hit_3 = (target_sid in top3_sids)
    
    borough_match_all = True
    if q["explicit_borough"]:
        borough_match_all = all(g["borough"] == q["explicit_borough"] for g in retrieved_groups)
    
    results_b.append({
        "qid": qid,
        "explicit_borough": q["explicit_borough"] or "—",
        "target": target_slug,
        "retrieved_top1": top1["shape_slug"] if top1 else "NONE",
        "score": f"{top1['score']:.4f}" if top1 else "—",
        "hit@1": "PASS" if hit_1 else "FAIL",
        "hit@3": "PASS" if hit_3 else "FAIL",
        "borough_filter_ok": "YES" if borough_match_all else "NO"
    })

eval_b_df = pd.DataFrame(results_b)
print("=== Condition B (Decoupled Vector + Payload Filter) Evaluation Results ===\n")
print(eval_b_df.to_string(index=False))

b_hit1_rate = (eval_b_df["hit@1"] == "PASS").mean()
b_hit3_rate = (eval_b_df["hit@3"] == "PASS").mean()
print(f"\nCondition B Summary: Shape Hit@1 = {b_hit1_rate:.1%} ({b_hit1_rate*len(eval_b_df):.0f}/{len(eval_b_df)}) | Shape Hit@3 = {b_hit3_rate:.1%} ({b_hit3_rate*len(eval_b_df):.0f}/{len(eval_b_df)})")

In [10]:
compare_df = pd.DataFrame({
    "qid": eval_a_df["qid"],
    "borough": eval_a_df["explicit_borough"],
    "target": eval_a_df["target"],
    "A_top1": eval_a_df["retrieved_top1"],
    "A_hit@1": eval_a_df["hit@1"],
    "B_top1": eval_b_df["retrieved_top1"],
    "B_hit@1": eval_b_df["hit@1"],
    "B_hit@3": eval_b_df["hit@3"],
})

print("=== Side-by-Side Comparison: Condition A vs. Condition B ===\n")
print(compare_df.to_string(index=False))

print("\n--- Exploratory Pilot Findings (8 Queries) ---")
print(f"Condition A (Borough in text):                        Shape Hit@1 = {hit1_rate:.1%} | Shape Hit@3 = {hit3_rate:.1%}")
print(f"Condition B (Decoupled text + query preprocessing): Shape Hit@1 = {b_hit1_rate:.1%} | Shape Hit@3 = {b_hit3_rate:.1%}")

resolved_failures = ((eval_a_df["hit@1"] == "FAIL") & (eval_b_df["hit@1"] == "PASS")).sum()
print(f"\nFailures under Condition A resolved by Condition B: {resolved_failures} (Q3, Q5, Q7)")
print("Note on Q8: Query targeted BLDG_BOTH but retrieved BLDG_NO_HEAT under both conditions.")
print("The 50.0% -> 87.5% delta reflects the combined Condition B design (both index text and query preprocessing + filter).")

=== Side-by-Side Comparison: Condition A vs. Condition B ===

qid   borough       target       A_top1 A_hit@1       B_top1 B_hit@1 B_hit@3
 Q1         —  APT_NO_HEAT  APT_NO_HEAT    PASS  APT_NO_HEAT    PASS    PASS
 Q2         —   APT_NO_HOT   APT_NO_HOT    PASS   APT_NO_HOT    PASS    PASS
 Q3         —    BLDG_BOTH BLDG_NO_HEAT    FAIL    BLDG_BOTH    PASS    PASS
 Q4         — BLDG_NO_HEAT BLDG_NO_HEAT    PASS BLDG_NO_HEAT    PASS    PASS
 Q5  BROOKLYN  APT_NO_HEAT BLDG_NO_HEAT    FAIL  APT_NO_HEAT    PASS    PASS
 Q6     BRONX BLDG_NO_HEAT BLDG_NO_HEAT    PASS BLDG_NO_HEAT    PASS    PASS
 Q7    QUEENS   APT_NO_HOT  BLDG_NO_HOT    FAIL   APT_NO_HOT    PASS    PASS
 Q8 MANHATTAN    BLDG_BOTH BLDG_NO_HEAT    FAIL BLDG_NO_HEAT    FAIL    PASS

--- Exploratory Pilot Findings (8 Queries) ---
Condition A (Borough in text):                        Shape Hit@1 = 50.0% | Shape Hit@3 = 100.0%
Condition B (Decoupled text + query preprocessing): Shape Hit@1 = 87.5% | Shape Hit@3 = 100.0%

Fail

In [ ]:
# 1. Verify incident_zip is absent from collection-01 payload
sample_point = client_qdrant.retrieve(
    collection_name=COLLECTION_01,
    ids=[int(df["unique_key"].iloc[0])],
    with_vectors=True,
    with_payload=True
)[0]

assert "incident_zip" not in sample_point.payload, "CRITICAL: incident_zip leaked into collection-01 payload!"
print("Payload Security Check: PASSED ('incident_zip' is absent from collection-01)")

# 2. Comprehensive 4-Boundary Temporal Verification Test Suite
intake_ref = "2024-01-27T18:00:00"

test_payload_base = {
    "unique_key": "99999999",
    "shape_id": 1,
    "shape_slug": "APT_NO_HEAT",
    "complaint_type": "HEAT/HOT WATER",
    "location_type": "RESIDENTIAL BUILDING",
    "descriptor": "APARTMENT ONLY",
    "descriptor_2": "NO HEAT",
    "borough": "MANHATTAN",
    "created_date": "2024-01-27T10:00:00",
    "status": "Closed",
    "resolution_description": "The Department of HPD inspected..."
}

# Boundary 1: Closed strictly BEFORE intake
b1 = format_retrieved_case({**test_payload_base, "closed_date": "2024-01-27T16:30:00"}, intake_time=intake_ref)
assert b1["is_historical_evidence"] is True, "Boundary 1 failed: closed before intake must be eligible!"
assert b1["status"] == "Closed", "Boundary 1 failed: status must be populated!"
assert b1["closed_date"] == "2024-01-27T16:30:00", "Boundary 1 failed: closed_date must be populated!"
assert b1["resolution_description"] is not None, "Boundary 1 failed: resolution must be populated!"

# Boundary 2: Closed strictly AFTER intake
b2 = format_retrieved_case({**test_payload_base, "closed_date": "2024-01-27T19:00:00"}, intake_time=intake_ref)
assert b2["is_historical_evidence"] is False, "Boundary 2 failed: closed after intake must be ineligible!"
assert b2["status"] is None, "Boundary 2 failed: status must be masked to None!"
assert b2["closed_date"] is None, "Boundary 2 failed: closed_date must be masked to None!"
assert b2["resolution_description"] is None, "Boundary 2 failed: resolution must be masked to None!"

# Boundary 3: Closed EXACTLY AT intake (must be strictly prior: '<', not '<=')
b3 = format_retrieved_case({**test_payload_base, "closed_date": "2024-01-27T18:00:00"}, intake_time=intake_ref)
assert b3["is_historical_evidence"] is False, "Boundary 3 failed: closed exactly at intake must be ineligible!"
assert b3["status"] is None, "Boundary 3 failed: status must be masked to None!"
assert b3["closed_date"] is None, "Boundary 3 failed: closed_date must be masked to None!"
assert b3["resolution_description"] is None, "Boundary 3 failed: resolution must be masked to None!"

# Boundary 4: No closure date (open ticket: closed_date is None)
b4 = format_retrieved_case({**test_payload_base, "closed_date": None, "status": "Open"}, intake_time=intake_ref)
assert b4["is_historical_evidence"] is False, "Boundary 4 failed: open ticket (null closed_date) must be ineligible!"
assert b4["status"] is None, "Boundary 4 failed: status must be masked to None!"
assert b4["closed_date"] is None, "Boundary 4 failed: closed_date must be masked to None!"
assert b4["resolution_description"] is None, "Boundary 4 failed: resolution must be masked to None!"

print("\nTemporal 4-Boundary Verification: ALL 4 BOUNDARIES PASSED WITH ASSERTIONS")
print("  * Boundary 1 (Closed before intake):     ELIGIBLE (outcomes exposed)")
print("  * Boundary 2 (Closed after intake):      INELIGIBLE (outcomes masked)")
print("  * Boundary 3 (Closed exactly at intake): INELIGIBLE (outcomes masked)")
print("  * Boundary 4 (No closure date / open):   INELIGIBLE (outcomes masked)")

In [ ]:
from datetime import datetime, timezone

sidecar_path = REPO_ROOT / "data" / "indexes" / f"{COLLECTION_01}.meta.json"

metadata = {
    "built_at": datetime.now(timezone.utc).isoformat(),
    "collection": COLLECTION_01,
    "server": settings.qdrant_url,
    "points": int(server_count),
    "vectors": f"size={settings.embedding_dimensions} distance=Cosine",
    "embedding_model": settings.gemini_embed_model,
    "distinct_complaint_shapes": len(SHAPE_CATALOG),
    "embedded_fields": fields_b,
    "payload_fields": list(sample_point.payload.keys()),
    "excluded_fields": ["incident_zip"],
    "filter_fields": ["borough"],
    "built_from": raw_file.name,
    "pilot_evaluation": {
        "queries_tested": len(PILOT_QUERIES),
        "condition_a_shape_hit1": float(hit1_rate),
        "condition_b_shape_hit1": float(b_hit1_rate),
        "shape_hit3": float(b_hit3_rate),
        "borough_filter_match_rate": 1.0
    },
    "findings": [
        "The 798 intake records in the January 27, 2024 extract partition exactly into 6 canonical complaint shapes",
        "In an exploratory 8-query pilot, comparing the complete Condition A design (borough in text) to Condition B (decoupled vector + query preprocessing + borough payload filter) yielded 4/8 (50.0%) vs 7/8 (87.5%) Shape Hit@1; Shape Hit@3 was 8/8 (100.0%) under both designs",
        "The performance difference reflects the full pipeline comparison: separating borough from the query text avoided borough tokens overriding apartment vs building scope in Q5 and Q7",
        "Q8 missed at Hit@1 under both designs (retrieving BLDG_NO_HEAT instead of BLDG_BOTH), showing that fine-grained defect distinctions remain challenging regardless of borough handling",
        "Borough payload filtering guarantees only that returned cases match the requested borough; complaint shape relevance is evaluated separately via Shape Hit@K",
        "Historical resolution evidence requires strict temporal masking: outcome fields are exposed only when closed_date < intake_time"
    ]
}

with open(sidecar_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved sidecar to {sidecar_path.name} ({sidecar_path.stat().st_size:,} bytes)")
print("Findings logged in sidecar:")
for item in metadata["findings"]:
    print(f"  * {item}")